# MEC-1 — Motor Estadístico de Clubes v1

Primera capa aprobada del proyecto de predicción de fútbol.

**Objetivo:** construir un modelo histórico prepartido para clubes que prediga goles esperados, marcador más probable, marcador al descanso, probabilidades 1X2, over/under y ambos anotan.

**Diseño:** sin leakage temporal. Cada partido se predice usando solo información anterior a su fecha.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# PROJECT_ROOT robusto: funciona si ejecutas desde notebooks/ o desde la raíz del proyecto.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

## 1. Imports del pipeline

In [ ]:
from src.mec1_pipeline import (
    DEFAULT_COMPETITIONS,
    run_mec1_pipeline,
    read_parquet_gzip,
    parquet_gzip_path,
    ensure_project_dirs,
    build_scoreline_matrix_for_match,
)

## 2. Parámetros

Requiere token de football-data.org. Lo más limpio es guardarlo como variable de entorno `FOOTBALL_DATA_API_TOKEN`.

Puedes empezar con pocas ligas/temporadas para probar y luego ampliar.

In [ ]:
# Ligas iniciales recomendadas.
COMPETITIONS = ["PL", "PD", "SA", "BL1", "FL1", "DED", "PPL", "CL"]

# Temporadas por año de inicio.
# Ejemplo: 2025 representa temporada 2025/2026 donde aplique.
SEASONS = [2021, 2022, 2023, 2024, 2025]

# Si quieres forzar descarga nuevamente, cambia a True.
FORCE_EXTRACT = False

## 3. Ejecutar MEC-1 completo

Este bloque:
1. Descarga raw JSON.
2. Normaliza a Bronze.
3. Construye features Silver.
4. Entrena modelos Gold.
5. Evalúa backtesting temporal.
6. Exporta Excel.

In [ ]:
result = run_mec1_pipeline(
    project_root=PROJECT_ROOT,
    competitions=COMPETITIONS,
    seasons=SEASONS,
    test_size=0.20,
    force_extract=FORCE_EXTRACT,
)

result.keys()

## 4. Métricas globales

In [ ]:
result["metrics_global"]

## 5. Métricas por liga/competición

In [ ]:
result["metrics_by_comp"].sort_values("log_loss_1x2").reset_index(drop=True)

## 6. Ver predicciones del backtesting

In [ ]:
cols = [
    "utc_date", "competition_code", "home_team_name", "away_team_name",
    "ft_home_goals", "ft_away_goals", "actual_result",
    "lambda_ft_home_goals", "lambda_ft_away_goals", "lambda_total_goals",
    "ft_top_scoreline", "ft_top_scoreline_prob", "ft_top3_scorelines",
    "ft_prob_home_win", "ft_prob_draw", "ft_prob_away_win",
    "ft_prob_over_25", "ft_prob_btts", "predicted_result",
]
result["pred_test"][[c for c in cols if c in result["pred_test"].columns]].head(30)

## 7. Matriz de marcadores de un partido

Cambia `idx` para revisar cualquier partido del backtesting.

In [ ]:
idx = 0
row = result["pred_test"].iloc[idx]

matrix_ft = build_scoreline_matrix_for_match(row, half="ft", max_goals=8)
matrix_ht = build_scoreline_matrix_for_match(row, half="ht", max_goals=8)

display(matrix_ft.head(15))
display(matrix_ht.head(15))

## 8. Leer archivos `.gzip` generados

Los archivos son Parquet comprimidos con gzip y extensión `.gzip`.

In [ ]:
paths = result["paths"]

model_dataset_path = parquet_gzip_path(paths["gold"], "model_dataset_mec1")
predictions_path = parquet_gzip_path(paths["gold"], "predictions_test_mec1")

model_dataset = pd.read_parquet(model_dataset_path)
predictions = pd.read_parquet(predictions_path)

print(model_dataset_path)
print(predictions_path)
print(model_dataset.shape, predictions.shape)

## 9. Ubicación del Excel final

In [ ]:
print(result["excel_path"])